# Tuning & Experiment Tracking

**Week 8 -- CS 203 -- Software Tools and Techniques for AI**

Prof. Nipun Batra, IIT Gandhinagar

---

**Outline:**
1. Grid Search vs Random Search
2. Bayesian Optimization with Optuna
3. Nested Cross-Validation (the optimism gap)
4. AutoML with AutoGluon
5. PyTorch Reproducibility
6. W&B Experiment Tracking

In [ ]:
# Install dependencies (uncomment if needed)
# !pip install numpy matplotlib scikit-learn optuna torch wandb seaborn

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import (
    train_test_split, cross_val_score, GridSearchCV,
    RandomizedSearchCV, StratifiedKFold
)
from sklearn.metrics import accuracy_score
from scipy.stats import randint, uniform
import warnings
warnings.filterwarnings("ignore")

plt.style.use("seaborn-v0_8-whitegrid")
np.random.seed(42)

In [ ]:
# ---- Synthetic dataset (used throughout Parts 1-4) ----
X, y = make_classification(
    n_samples=1000, n_features=10, n_informative=5,
    n_redundant=2, n_classes=2, random_state=42
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(f"Dataset : {X.shape[0]} samples, {X.shape[1]} features")
print(f"Train   : {X_train.shape[0]}  |  Test: {X_test.shape[0]}")
print(f"Class balance: {y.mean():.1%} positive")

---
## Part 1: Grid Search vs Random Search

In [ ]:
# --- Grid Search ---
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10, 15, None],
    'min_samples_leaf': [1, 2, 5],
}

grid_cv = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    return_train_score=True,
)
grid_cv.fit(X_train, y_train)

BUDGET = len(grid_cv.cv_results_['params'])  # total grid combos
print(f"Grid Search: {BUDGET} combinations evaluated")
print(f"Best CV score : {grid_cv.best_score_:.4f}")
print(f"Best params   : {grid_cv.best_params_}")

In [ ]:
# --- Random Search (same budget) ---
param_distributions = {
    'n_estimators': randint(50, 500),
    'max_depth': randint(3, 30),
    'min_samples_leaf': randint(1, 20),
    'max_features': uniform(0.1, 0.9),
}

random_cv = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    param_distributions,
    n_iter=BUDGET,
    cv=5,
    scoring='accuracy',
    random_state=42,
    n_jobs=-1,
    return_train_score=True,
)
random_cv.fit(X_train, y_train)

print(f"Random Search: {BUDGET} combinations evaluated")
print(f"Best CV score : {random_cv.best_score_:.4f}")
print(f"Best params   : {random_cv.best_params_}")
print()
diff = random_cv.best_score_ - grid_cv.best_score_
winner = "Random" if diff > 0 else "Grid"
print(f"{winner} Search wins by {abs(diff):.4f}")

In [ ]:
# --- Plot: parameter space explored by each method ---
grid_params = grid_cv.cv_results_['params']
rand_params = random_cv.cv_results_['params']

def extract(params_list, key, default=None):
    """Extract a hyperparameter from a list of param dicts."""
    vals = []
    for p in params_list:
        v = p.get(key, default)
        vals.append(v if v is not None else 30)  # map None -> 30 for plotting
    return np.array(vals, dtype=float)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Grid search explored space
ax = axes[0]
g_ne = extract(grid_params, 'n_estimators')
g_md = extract(grid_params, 'max_depth')
g_scores = grid_cv.cv_results_['mean_test_score']
sc = ax.scatter(g_ne, g_md, c=g_scores, cmap='viridis', s=60, edgecolors='k', linewidths=0.5)
ax.set_xlabel('n_estimators')
ax.set_ylabel('max_depth (None -> 30)')
ax.set_title(f'Grid Search ({BUDGET} evals)')
plt.colorbar(sc, ax=ax, label='CV Accuracy')

# Random search explored space
ax = axes[1]
r_ne = extract(rand_params, 'n_estimators')
r_md = extract(rand_params, 'max_depth')
r_scores = random_cv.cv_results_['mean_test_score']
sc = ax.scatter(r_ne, r_md, c=r_scores, cmap='viridis', s=60, edgecolors='k', linewidths=0.5)
ax.set_xlabel('n_estimators')
ax.set_ylabel('max_depth')
ax.set_title(f'Random Search ({BUDGET} evals)')
plt.colorbar(sc, ax=ax, label='CV Accuracy')

fig.suptitle('Parameter Space Explored: Grid vs Random', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# --- Plot: min_samples_leaf vs max_depth (second view) ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
g_ml = extract(grid_params, 'min_samples_leaf')
ax.scatter(g_ml, g_md, c=g_scores, cmap='viridis', s=60, edgecolors='k', linewidths=0.5)
ax.set_xlabel('min_samples_leaf')
ax.set_ylabel('max_depth (None -> 30)')
ax.set_title('Grid Search')

ax = axes[1]
r_ml = extract(rand_params, 'min_samples_leaf')
sc = ax.scatter(r_ml, r_md, c=r_scores, cmap='viridis', s=60, edgecolors='k', linewidths=0.5)
ax.set_xlabel('min_samples_leaf')
ax.set_ylabel('max_depth')
ax.set_title('Random Search')
plt.colorbar(sc, ax=axes, label='CV Accuracy')

fig.suptitle('Random covers more of the continuous space', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

**Takeaway:** Grid search only evaluates points on a fixed lattice. Random search covers the space more uniformly, which matters when some hyperparameters are more important than others (Bergstra & Bengio, 2012).

---
## Part 2: Optuna (Bayesian Optimization)

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 30),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 20),
        'max_features': trial.suggest_float('max_features', 0.1, 1.0),
    }
    model = RandomForestClassifier(**params, random_state=42)
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy', n_jobs=-1)
    return scores.mean()

study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=BUDGET)

print(f"Optuna: {BUDGET} trials")
print(f"Best CV score : {study.best_value:.4f}")
print(f"Best params   : {study.best_params}")

In [ ]:
# Optuna built-in plot: optimization history
fig = optuna.visualization.plot_optimization_history(study)
fig.show()

In [ ]:
# Optuna built-in plot: parameter importances
fig = optuna.visualization.plot_param_importances(study)
fig.show()

In [ ]:
# --- Compare all three methods ---
methods = ['Grid Search', 'Random Search', 'Optuna (TPE)']
scores = [grid_cv.best_score_, random_cv.best_score_, study.best_value]

print("=" * 50)
print(f"{'Method':<20} {'Best CV Accuracy':>15}")
print("=" * 50)
for m, s in zip(methods, scores):
    print(f"{m:<20} {s:>15.4f}")
print("=" * 50)
best_idx = int(np.argmax(scores))
print(f"Winner: {methods[best_idx]}")

# Bar chart comparison
fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#4C72B0', '#55A868', '#C44E52']
bars = ax.bar(methods, scores, color=colors, edgecolor='black', linewidth=0.8)
ax.set_ylabel('Best CV Accuracy')
ax.set_title(f'Tuning Methods Comparison (Budget = {BUDGET} evaluations each)')
ax.set_ylim(min(scores) - 0.01, max(scores) + 0.005)
for bar, score in zip(bars, scores):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.001,
            f'{score:.4f}', ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.show()

---
## Part 3: Nested Cross-Validation

`GridSearchCV.best_score_` is **optimistic** because it reports the best score found after searching -- this is selection bias. Nested CV gives an unbiased estimate by evaluating the entire tuning procedure in an outer loop.

In [ ]:
# --- The optimism gap ---

# Inner CV: tunes hyperparameters
inner_cv = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid={'max_depth': [5, 10, 15, None], 'n_estimators': [50, 100, 200]},
    cv=3,
    scoring='accuracy',
    n_jobs=-1,
)

# Outer CV: evaluates the tuned pipeline
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
outer_scores = cross_val_score(inner_cv, X, y, cv=outer_cv, scoring='accuracy')

# Also get the (optimistic) best_score_ by fitting on full data
inner_cv.fit(X, y)
optimistic_score = inner_cv.best_score_
nested_score = outer_scores.mean()
gap = optimistic_score - nested_score

print("=" * 55)
print(f"  GridSearchCV.best_score_ (OPTIMISTIC) : {optimistic_score:.4f}")
print(f"  Nested CV outer scores                : {outer_scores}")
print(f"  Nested CV mean (UNBIASED)             : {nested_score:.4f} +/- {outer_scores.std():.4f}")
print(f"  Optimism gap                          : {gap:.4f}")
print("=" * 55)
print()
print("The gap shows how much GridSearchCV.best_score_ overestimates")
print("true generalization performance due to selection bias.")

In [ ]:
# --- Visualize the gap ---
fig, ax = plt.subplots(figsize=(8, 4))

ax.barh(['Nested CV\n(unbiased)', 'GridSearchCV.best_score_\n(optimistic)'],
        [nested_score, optimistic_score],
        color=['#55A868', '#C44E52'], edgecolor='black', linewidth=0.8, height=0.5)

# Error bar for nested CV
ax.errorbar(nested_score, 0, xerr=outer_scores.std(), fmt='none',
            ecolor='black', capsize=5, linewidth=2)

ax.set_xlabel('Accuracy')
ax.set_title(f'The Optimism Gap: {gap:.4f}')
ax.set_xlim(min(nested_score - outer_scores.std(), optimistic_score) - 0.02,
            max(nested_score, optimistic_score) + 0.02)

for i, v in enumerate([nested_score, optimistic_score]):
    ax.text(v + 0.003, i, f'{v:.4f}', va='center', fontweight='bold')

plt.tight_layout()
plt.show()

---
## Part 4: AutoML with AutoGluon

AutoGluon automates model selection, hyperparameter tuning, and ensembling. It is a heavy dependency, so the code below is **commented out** with instructions.

In [ ]:
# ============================================================
# AutoGluon: 3-line AutoML  (commented out -- heavy dependency)
# ============================================================
# To run this cell:
#   1. pip install autogluon
#   2. Uncomment the code below
#   3. Run the cell
# ============================================================

# import pandas as pd
# from autogluon.tabular import TabularPredictor
#
# # Wrap our numpy arrays in a DataFrame for AutoGluon
# train_df = pd.DataFrame(X_train, columns=[f'f{i}' for i in range(X_train.shape[1])])
# train_df['target'] = y_train
# test_df = pd.DataFrame(X_test, columns=[f'f{i}' for i in range(X_test.shape[1])])
# test_df['target'] = y_test
#
# # --- The 3 lines ---
# predictor = TabularPredictor(label='target', eval_metric='accuracy')
# predictor.fit(train_df, time_limit=120)        # train for 2 minutes
# leaderboard = predictor.leaderboard(test_df)    # evaluate all models
# print(leaderboard)

print("AutoGluon code is commented out (heavy dependency).")
print("Below is a mock leaderboard showing typical AutoGluon output.")

In [ ]:
# --- Mock AutoGluon leaderboard (representative output) ---
import pandas as pd

mock_leaderboard = pd.DataFrame({
    'model': [
        'WeightedEnsemble_L2',
        'LightGBM',
        'CatBoost',
        'RandomForestGini',
        'ExtraTreesGini',
        'XGBoost',
        'NeuralNetFastAI',
        'KNeighborsDist',
    ],
    'score_val': [0.9475, 0.9425, 0.9400, 0.9350, 0.9325, 0.9300, 0.9200, 0.8950],
    'score_test': [0.9400, 0.9350, 0.9300, 0.9250, 0.9250, 0.9200, 0.9100, 0.8850],
    'fit_time': [58.2, 3.1, 8.7, 2.4, 1.9, 4.5, 22.1, 0.3],
    'pred_time_val': [0.12, 0.02, 0.03, 0.05, 0.04, 0.02, 0.08, 0.15],
})
print("Mock AutoGluon Leaderboard (typical output):")
print(mock_leaderboard.to_string(index=False))

In [ ]:
# --- Complete workflow: Dummy -> LR -> Tuned RF -> AutoML ---
# This shows the natural progression a practitioner should follow.

results = {}

# 1. Dummy baseline
dummy = DummyClassifier(strategy='most_frequent', random_state=42)
dummy.fit(X_train, y_train)
results['Dummy (baseline)'] = dummy.score(X_test, y_test)

# 2. Logistic Regression (simple model)
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)
results['Logistic Regression'] = lr.score(X_test, y_test)

# 3. Tuned Random Forest (from our grid/random/optuna search)
best_rf = grid_cv.best_estimator_
results['Tuned RF (GridSearch)'] = best_rf.score(X_test, y_test)

# 4. AutoML (mock -- use the top line from leaderboard)
results['AutoML (AutoGluon)*'] = 0.9400  # from mock leaderboard above

print(f"{'Model':<25} {'Test Accuracy':>15}")
print("=" * 42)
for name, acc in results.items():
    marker = ' <-- mock' if '*' in name else ''
    print(f"{name:<25} {acc:>15.4f}{marker}")

# Plot
fig, ax = plt.subplots(figsize=(9, 4))
names = list(results.keys())
accs = list(results.values())
colors = ['#cccccc', '#4C72B0', '#55A868', '#C44E52']
bars = ax.barh(names, accs, color=colors, edgecolor='black', linewidth=0.8)
ax.set_xlabel('Test Accuracy')
ax.set_title('Complete Workflow: Baselines through AutoML')
ax.set_xlim(0.4, 1.0)
for bar, acc in zip(bars, accs):
    ax.text(acc + 0.005, bar.get_y() + bar.get_height() / 2,
            f'{acc:.4f}', va='center', fontweight='bold')
plt.tight_layout()
plt.show()

---
## Part 5: PyTorch Reproducibility

In [ ]:
import torch
import torch.nn as nn
import random
import os

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")

In [ ]:
def set_seed_full(seed=42):
    """Complete seed function for full PyTorch reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    torch.use_deterministic_algorithms(True)

In [ ]:
# --- Without seeds: different output every run ---
print("=== WITHOUT seeds (outputs differ each run) ===")
outputs_no_seed = []
for i in range(3):
    model = nn.Linear(10, 1)
    x = torch.randn(5, 10)
    y_pred = model(x)
    val = y_pred[0].item()
    outputs_no_seed.append(val)
    print(f"  Run {i+1}: first output = {val:.6f}")

print(f"  All same? {len(set([f'{v:.6f}' for v in outputs_no_seed])) == 1}")

In [ ]:
# --- With set_seed_full: identical output every run ---
print("=== WITH set_seed_full (outputs are identical) ===")
outputs_seeded = []
for i in range(3):
    set_seed_full(42)
    model = nn.Linear(10, 1)
    x = torch.randn(5, 10)
    y_pred = model(x)
    val = y_pred[0].item()
    outputs_seeded.append(val)
    print(f"  Run {i+1}: first output = {val:.6f}")

print(f"  All same? {len(set([f'{v:.6f}' for v in outputs_seeded])) == 1}")

In [ ]:
# --- DataLoader reproducibility with worker_init_fn ---
from torch.utils.data import DataLoader, TensorDataset

def seed_worker(worker_id):
    """Ensure each DataLoader worker gets a deterministic seed."""
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

print("=== DataLoader reproducibility with worker_init_fn ===")
batch_sums = []
for i in range(3):
    set_seed_full(42)
    g = torch.Generator()
    g.manual_seed(42)

    dataset = TensorDataset(
        torch.randn(100, 10),
        torch.randint(0, 2, (100,))
    )
    loader = DataLoader(
        dataset, batch_size=16, shuffle=True,
        worker_init_fn=seed_worker,
        generator=g,
    )
    first_batch_x, first_batch_y = next(iter(loader))
    s = first_batch_x.sum().item()
    batch_sums.append(s)
    print(f"  Run {i+1}: batch sum = {s:.4f}, labels = {first_batch_y[:5].tolist()}")

print(f"  All same? {len(set([f'{v:.4f}' for v in batch_sums])) == 1}")

In [ ]:
# --- Multi-seed reporting ---
# Better than a single deterministic run: report mean +/- std across seeds.

print("=== Multi-Seed Reporting ===")
seed_results = []
seeds = [42, 123, 456, 789, 1024]

for seed in seeds:
    np.random.seed(seed)
    X_s, y_s = make_classification(
        n_samples=500, n_features=10, n_informative=5,
        n_redundant=2, random_state=seed
    )
    Xtr, Xte, ytr, yte = train_test_split(X_s, y_s, test_size=0.2, random_state=seed)
    m = RandomForestClassifier(n_estimators=100, random_state=seed)
    m.fit(Xtr, ytr)
    acc = m.score(Xte, yte)
    seed_results.append(acc)
    print(f"  Seed {seed:>4d}: accuracy = {acc:.4f}")

mean_acc = np.mean(seed_results)
std_acc = np.std(seed_results)
print(f"\nResult: {mean_acc:.4f} +/- {std_acc:.4f}")
print("This is more honest than reporting a single lucky seed.")

# Plot
fig, ax = plt.subplots(figsize=(8, 3))
ax.barh([f'Seed {s}' for s in seeds], seed_results, color='#4C72B0',
        edgecolor='black', linewidth=0.5)
ax.axvline(mean_acc, color='red', linestyle='--', linewidth=2, label=f'Mean = {mean_acc:.4f}')
ax.set_xlabel('Test Accuracy')
ax.set_title('Multi-Seed Reporting')
ax.legend()
plt.tight_layout()
plt.show()

---
## Part 6: W&B Integration

Weights & Biases (W&B) tracks experiments: hyperparameters, metrics, code versions, and artifacts.

**Instructions for students:**
1. Create a free account at [wandb.ai](https://wandb.ai)
2. `pip install wandb`
3. `wandb login` (paste your API key)
4. Uncomment the code below and run it

In [ ]:
# ============================================================
# W&B Integration  (uncomment to run with your own API key)
# ============================================================
# import wandb
#
# # --- Initialize a run ---
# wandb.init(
#     project="cs203-week08-demo",
#     config={
#         "model": "RandomForest",
#         "n_estimators": 200,
#         "max_depth": 10,
#         "min_samples_leaf": 2,
#         "dataset_size": X_train.shape[0],
#         "n_features": X_train.shape[1],
#         "seed": 42,
#     }
# )
#
# # --- Train the model ---
# model = RandomForestClassifier(
#     n_estimators=wandb.config.n_estimators,
#     max_depth=wandb.config.max_depth,
#     min_samples_leaf=wandb.config.min_samples_leaf,
#     random_state=wandb.config.seed,
# )
# model.fit(X_train, y_train)
#
# # --- Log metrics ---
# train_acc = model.score(X_train, y_train)
# test_acc = model.score(X_test, y_test)
# wandb.log({
#     "train_accuracy": train_acc,
#     "test_accuracy": test_acc,
#     "train_test_gap": train_acc - test_acc,
# })
#
# # --- Log feature importances ---
# for i, imp in enumerate(model.feature_importances_):
#     wandb.log({f"feature_{i}_importance": imp})
#
# # --- Log a matplotlib figure ---
# fig, ax = plt.subplots(figsize=(8, 4))
# ax.bar(range(len(model.feature_importances_)), model.feature_importances_)
# ax.set_xlabel('Feature index')
# ax.set_ylabel('Importance')
# ax.set_title('Feature Importances')
# wandb.log({"feature_importances_plot": wandb.Image(fig)})
# plt.close(fig)
#
# print(f"Train accuracy: {train_acc:.4f}")
# print(f"Test accuracy : {test_acc:.4f}")
# print(f"View your run at: {wandb.run.url}")
# wandb.finish()

print("W&B code is commented out.")
print("Uncomment and run with your own API key to see:")
print("  - Hyperparameters logged automatically")
print("  - Train/test accuracy tracked")
print("  - Feature importance plots stored")
print("  - Interactive dashboard at wandb.ai")

---
## Summary

| Topic | Key takeaway |
|-------|-------------|
| Grid vs Random | Random covers the space better with same budget |
| Optuna | Bayesian optimization learns from past trials, often finds better configs |
| Nested CV | `best_score_` is optimistic; nested CV is unbiased |
| AutoGluon | 3-line AutoML that handles model selection + ensembling |
| Reproducibility | `set_seed_full` + `worker_init_fn` + deterministic algorithms |
| Multi-seed | Report mean +/- std across seeds, not a single lucky run |
| W&B | Tracks params, metrics, code, artifacts in a dashboard |